## 命名实体识别
NER(Named Entiry Recognition), NER是自然语言处理中的一项基础任务，旨在从文本中识别和分类出具有特定意义的命名实体(Named Entiry)。

**spaCy命名实体识别的实体类型：**

类型  | 描述 
--- | ---
PERSON | 人名，包括虚构人名
NORP | 名族、宗教或者政治团体
FAC | 建筑、机场、高速公路和桥梁等
ORG | 公司、政府部门和机构等
GPE | 国家、城市和州
LOC | 非GPE类的位置、山脉和流域
PRODUCT | 实物、车辆和实物等（非服务类）
EVENT | 命名飓风、战斗、战争和体育赛事等
WORK_OF_ART | 书名、歌单等
LAW | 成文法律的具名文件
LANGUAGE | 任何命名语言
DATE | 绝对/相对的日期/时间段
TIME | 小于一天的时间
PERCENT | 百分比，包括`%`符号
MONEY | 货币值，包括单位
QUANTITY | 计量，比如重量或距离
ORDINAL | 序数，比如“第一”“第二”等
CARDINAL | 不属于其它类别的基数或者纯数字

In [1]:
import torch
import pprint
import pandas as pd

In [2]:
# 去除一些警告信息
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### 1. 准备数据
我们先准备好AG-News数据集，参考文档：[kaggle-AG-News数据集](../12-datasets/31-kaggle-AG-News数据集.ipynb)。得到`train_data.csv`文件。

In [3]:
AG_DATASET_DIR = "../../data/kaggle/AG-News-Classfication-Dataset"

In [4]:
# 查看数据
!ls {AG_DATASET_DIR}

test.csv       train.csv      train_data.csv


In [5]:
# 加载数据集
data = pd.read_csv(f"{AG_DATASET_DIR}/train_data.csv")

In [6]:
data.head()

,class_index,title,description,class_name
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",商业
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,商业
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries ab...,商业
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export f...,商业
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...",商业


> 我么使用train_data.csv来练习NER任务。

### 2. 使用spacy识别命名实体

#### 2.1 加载模型

In [7]:
# 使用gpu
import spacy

In [8]:
# 尝试将spaCy的处理移置GPU上
spacy.require_gpu()

True

In [9]:
# 加载预训练模型之前先下载
# !python -m spacy download en_core_web_trf

In [10]:
# 加载预训练的模型: 
# 如果报错：`Can't find model 'en_core_web_trf'. It doesn't seem to be a Python package or a valid path to a data directory.`
# 请先执行模型下载：python -m spacy download en_core_web_trf
nlp = spacy.load("en_core_web_trf")

In [11]:
# 打印模型的元信息
pp = pprint.PrettyPrinter(indent=2)

In [12]:
# pp.pprint(nlp.meta)

#### 2.2 识别新闻数据中的命名实体

In [13]:
description = data.loc[0, "description"]
description

"Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again."

In [14]:
nlp

In [15]:
doc = nlp(description)
type(doc)

spacy.tokens.doc.Doc

In [16]:
doc.ents

(Reuters,)

In [17]:
type(doc.ents)

tuple

In [18]:
ent = doc.ents[0]
# 命名实体的文本、开始位置、结束位置、标签
ent.text, ent.start_char, ent.end_char, ent.label_

('Reuters', 0, 7, 'ORG')

#### 2.3 批量识别数据集中N条描述的命名实体

In [19]:
for i in range(8, 13):
    description = data.loc[i, "description"]
    print("{:2}:\t{}".format(i, description))
    # 模型call
    doc = nlp(description)
    # 遍历输出命名实体信息：Text, Start, End, Label
    for ent in doc.ents:
        print("\t", ent.text, ent.start_char, ent.end_char, ent.label_)
    # 换一行
    print("\n")

 8:	Forbes.com - After earning a PH.D. in Sociology, Danny Bazil Riley started to work as the general manager at a commercial real estate firm at an annual base salary of $;70,000. Soon after, a financial planner stopped by his desk to drop off brochures about insurance benefits available through his employer. But, at 32, "buying insurance was the furthest thing from my mind," says Riley.
	 Forbes.com 0 10 ORG
	 PH.D. 29 34 WORK_OF_ART
	 Sociology 38 47 ORG
	 Danny Bazil Riley 49 66 PERSON
	 $;70,000 167 175 MONEY
	 32 316 318 DATE
	 Riley 381 386 PERSON


 9:	NEW YORK (Reuters) - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.
	 NEW YORK 0 8 GPE
	 Reuters 10 17 ORG


10:	NEW YORK (Reuters) - Soaring crude prices plus worries about the economy and the outlook for earnings are expected to hang over the stock market next week during the depth of the summer doldrums.
	 NEW YORK 0 8 GPE
	 Reuters 10 17 ORG
	 next week 145 154 DATE
	 summer 179 185 DATE
